<a href="https://colab.research.google.com/github/viet-coding/brew-watch/blob/main/260421__Cluster_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Highlands Coffee - Cluster Analysis (500m)

**Counts how many competitor stores sit within 500m of each Highlands location.**

Input : `260421_Highlands_Store_geoloc_vshared.xlsx` (~990 active stores)
Output: `Highlands_Competitor_500m.xlsx` with two sheets:
- **Summary**: one row per Highlands store, one column per competitor with count
- **Details**: one row per nearby competitor match (for verification / drill-down)

**Competitors tracked** (7): Starbucks, The Coffee House, Phe La, Katinat, Cong, Trung Nguyen Legend, Phuc Long

## Expected runtime & cost (500m, 1,008 stores, 7 competitors + self-check)

- API calls: 1,008 × 8 = **~8,064 base calls**, ~10,000 with pagination in dense urban stores
- Billing SKU: **Nearby Search Pro** (Places API New)
  - First **5,000 calls/month**: free (per billing account, shared across all projects)
  - Calls beyond 5,000: **\$32 / 1,000**
- **Net out-of-pocket ≈ \$160** — this is a real charge, not offset by any credit
- Under the 2025 pricing model, the old universal \$200 monthly credit no longer applies. Each SKU has its own free quota instead
- Watch out: if your team has other Nearby Search Pro usage this month, less of the 5,000 free tier is available for this job — pushing the bill higher
- Re-running the script adds another ~\$160 at full price (no more free quota left for the month)
- Billing posts with a **24–72 hour lag** on the dashboard
- Runtime: ~40–45 minutes at 0.25s/call

## 1. Install & import

In [3]:
!pip install -q requests pandas openpyxl unidecode tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.2 MB/s eta 0:00:00


In [4]:
import math
import pickle
import time
import re
from pathlib import Path

import pandas as pd
import requests
from tqdm.auto import tqdm
from unidecode import unidecode


## 2. Update your API key and file name here; adjust other settings if needed

In [16]:
# ---- API key ----------------------------------------------------------------
API_KEY = 'API_KEY'  # paste your key here

# ---- Analysis parameters ----------------------------------------------------
RADIUS_M = 500                     # search radius in metres

# Competitor brand -> list of name variants to match against
# (unidecode is applied to both sides before comparison, so accented forms work)
COMPETITORS = {
    "Starbucks":            ["Starbucks"],
    "The Coffee House":     ["The Coffee House", "Coffee House"],
    "Phe La":               ["Phe La", "Phê La"],
    "Katinat":              ["Katinat"],
    "Cong":                 ["Cong Caphe", "Cong Ca Phe", "Công Cà Phê", "Cộng Cà Phê"],
    "Trung Nguyen Legend":  ["Trung Nguyen Legend", "Trung Nguyên Legend"],
    "Phuc Long":            ["Phuc Long", "Phúc Long"],
}

# Whether to also look up each Highlands on Google (will add more cost).
# Set True if you want rating / reviews / hours / maps URL for each store.
INCLUDE_STORE_LOOKUP = False

# ---- Files ------------------------------------------------------------------
INPUT_FILE = "260421_Highlands_Store geoloc_vshared.xlsx"
CHECKPOINT_FILE = "checkpoint_500m.pkl"
OUTPUT_FILE     = "Highlands_Competitor_500m.xlsx"

# ---- Pacing -----------------------------------------------------------------
SLEEP_BETWEEN_CALLS = 0.2          # seconds; raise if you hit rate limits
CHECKPOINT_EVERY    = 25           # save progress every N stores

print("Config loaded.")
print(f"  Radius:       {RADIUS_M} m")
print(f"  Competitors:  {len(COMPETITORS)}")
print(f"  Store lookup: {INCLUDE_STORE_LOOKUP}")


Config loaded.
  Radius:       500 m
  Competitors:  7
  Store lookup: False


## 3. Upload the Highlands store excel with coordinate data

In [6]:
from google.colab import files
uploaded = files.upload()   # pick 260421_Highlands_Store_geoloc_vshared.xlsx
print("✅ Uploaded:", list(uploaded.keys()))


Saving 260421_Highlands_Store geoloc_vshared.xlsx to 260421_Highlands_Store geoloc_vshared.xlsx
✅ Uploaded: ['260421_Highlands_Store geoloc_vshared.xlsx']


## 4. Load & validate the input

This cell surfaces three common issues so you can decide how to handle them:

1. **Missing coordinates** — auto-skipped (can't process without lat/lng)
2. **Duplicate coordinate pairs** — two rows at the same spot. By default
   we keep both and let each be scored independently. Set `DROP_COORD_DUPES = True`
   below if you'd rather collapse them.
3. **Final count** — reported explicitly so you can confirm it matches your
   expected active-store list (~990).

If the final count is off, pre-filter the Excel before uploading rather
than patching logic here.

In [7]:
# Toggle: collapse stores that share identical lat/lng into a single row?
DROP_COORD_DUPES = False

df = pd.read_excel(INPUT_FILE)
print(f"Loaded {len(df):,} rows from file")
print("Columns:", df.columns.tolist())

# ---- 1. Missing coordinates -------------------------------------------------
missing_coords = df[df["Latitude"].isna() | df["Longitude"].isna()]
if len(missing_coords):
    print(f"\n⚠️  {len(missing_coords)} rows missing lat/lng — will be skipped:")
    print(missing_coords[["Store", "Address"]].to_string(index=False))

df = df.dropna(subset=["Latitude", "Longitude"]).reset_index(drop=True)

# ---- 2. Duplicate coordinate pairs -----------------------------------------
coord_dupes = df[df.duplicated(subset=["Latitude", "Longitude"], keep=False)]
coord_dupes = coord_dupes.sort_values(["Latitude", "Longitude"])
if len(coord_dupes):
    print(f"\n⚠️  {len(coord_dupes)} rows share lat/lng with another row "
          f"({coord_dupes[['Latitude','Longitude']].drop_duplicates().shape[0]} unique coord pairs):")
    print(coord_dupes[["Store", "NS code only", "Latitude", "Longitude"]].to_string(index=False))
    if DROP_COORD_DUPES:
        before = len(df)
        df = df.drop_duplicates(subset=["Latitude", "Longitude"], keep="first").reset_index(drop=True)
        print(f"\n  DROP_COORD_DUPES is True — removed {before - len(df)} duplicate rows")
    else:
        print("\n  DROP_COORD_DUPES is False — keeping all rows "
              "(set True above to collapse duplicates)")

# ---- 3. Final count --------------------------------------------------------
print(f"\n✅ {len(df):,} stores ready for processing")
print("   → If this number doesn't match your expected active-store count, "
      "pre-filter the input Excel before rerunning.")
df.head()


Loaded 1,008 rows from file
Columns: ['Store', 'NS code only', 'Address', 'Latitude', 'Longitude']

⚠️  12 rows missing lat/lng — will be skipped:
                                           Store                                                                         Address
          HCSDNA0051 A-04 Vo Nguyen Giap Da Nang              Slot A-04 Vo Nguyen Giap St, Phuoc My W, \nSon Tra D, Da Nang City
                     HCSDNA0054 Go! Mall Da Nang                                       257 Hùng Vương, phường Thanh Khê, Đà Nẵng
                    HCSDNA0052 ThaiGrand Da Nang                        Bach Dang St, Hoa Thuan Dong W, Hai Chau D, Da Nang City
                 HCSHNI0238 Ben tau song Hong HN                       46 P. Chương Dương Độ, Chương Dương Độ, Hoàn Kiếm, Hà Nội
                   HCSHNI0236 Viglacera Tower HN Viglacera Tower, No.1 Thang Long Boulevard, Me Tri W., Nam Tu Liem Dist., Hanoi
                           HCSHCM0362 Bitexco D1                               

,Store,NS code only,Address,Latitude,Longitude
0,HCSHCM0005 Saigon South,HCSHCM0005,"Gian hàng 3&4 tầng trệt Broadway D, Số 152 Ngu...",10.728923,106.721352
1,HCSHCM0006 Saigon Trade,HCSHCM0006,"37 Đ. Tôn Đức Thắng, Bến Nghé, Quận 1, Hồ Chí ...",10.784053,106.703452
2,HCSHNI0001 Ho Guom Plaza,HCSHNI0001,"Tầng 3, tòa nhà Hồ Gươm Plaza, số 1-3-5 phố Đi...",20.979162,105.785657
3,HCSHNI0002 Big C Ha Noi,HCSHNI0002,"Gian hàng số 22, TTTM Bourbon Thăng Long, số 2...",21.006929,105.793557
4,HCSHNI0003 Flag Tower,HCSHNI0003,"28A Dien Bien Phu Street, Ba Dinh Dist, HN",21.032318,105.839792


## 5. Helpers — distance & strict name matching

- `haversine_distance` — great-circle distance in metres.
- `normalize` — lowercases and strips Vietnamese diacritics (`phê` → `phe`).
- `name_matches_brand` — true only if a normalized brand variant appears as
  a distinct substring in the normalized place name. Prevents `"Phuc Long"`
  from matching an unrelated shop that happens to contain those letters.

In [8]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """Great-circle distance in metres."""
    R = 6_371_000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def normalize(text):
    """Lowercase + strip diacritics + collapse whitespace."""
    if not text:
        return ""
    return re.sub(r"\s+", " ", unidecode(text).lower()).strip()


def name_matches_brand(place_name, brand_variants):
    """
    True iff the normalized place_name contains any of the normalized brand
    variants as a distinct token sequence (word-boundaried).
    """
    norm_name = normalize(place_name)
    for variant in brand_variants:
        norm_variant = normalize(variant)
        # word-boundary match — prevents "congee" matching "cong"
        if re.search(r"\b" + re.escape(norm_variant) + r"\b", norm_name):
            return True
    return False


# Sanity checks
assert name_matches_brand("Phúc Long Coffee & Tea", ["Phuc Long", "Phúc Long"])
assert name_matches_brand("Katinat Saigon Kafe", ["Katinat"])
assert not name_matches_brand("Con Gà Vàng Restaurant", ["Cong Caphe"])
assert not name_matches_brand("Long Phuc Massage", ["Phuc Long"])  # word order matters
print("✅ Helper tests pass")


✅ Helper tests pass


## 6. Nearby Search with pagination + strict matching

**Why pagination matters at 500m.** Google's Nearby Search returns at most
20 places per call. In dense districts a single keyword query can easily
exceed that. Google provides a `next_page_token` to fetch the next 20
(up to 60 total). We loop through them and stop early when we've fetched
all pages.

**The function returns matches, not a bare count**, so the Details sheet
can show *which* specific stores were counted.

In [9]:
NEARBY_URL = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"


def nearby_brand_matches(lat, lng, radius_m, keyword, variants):
    """
    Find places near (lat,lng) within radius_m that genuinely match the brand.

    Returns a list of dicts: [{name, vicinity, lat, lng, distance_m, place_id}, ...]
    """
    matches = []
    seen_place_ids = set()
    params = {
        "location": f"{lat},{lng}",
        "radius":   radius_m,
        "keyword":  keyword,
        "type":     "cafe",          # drops hotels, malls, unrelated shops
        "key":      API_KEY,
    }

    page = 0
    while True:
        r = requests.get(NEARBY_URL, params=params).json()
        status = r.get("status", "")

        # INVALID_REQUEST on a pagetoken usually means the token isn't ready
        # yet — Google requires a short delay after issuing next_page_token.
        if status == "INVALID_REQUEST" and "pagetoken" in params:
            time.sleep(2)
            continue
        if status not in ("OK", "ZERO_RESULTS"):
            # Surface anything weird (OVER_QUERY_LIMIT, REQUEST_DENIED, etc.)
            print(f"  ⚠️  Nearby Search status={status} for {keyword}: "
                  f"{r.get('error_message', '')}")
            break

        for place in r.get("results", []):
            pid = place.get("place_id")
            if pid in seen_place_ids:
                continue
            seen_place_ids.add(pid)

            name = place.get("name", "")
            if not name_matches_brand(name, variants):
                continue

            loc = place.get("geometry", {}).get("location", {})
            if not loc:
                continue
            dist = haversine_distance(lat, lng, loc["lat"], loc["lng"])
            if dist > radius_m:
                continue

            matches.append({
                "name":       name,
                "vicinity":   place.get("vicinity", ""),
                "lat":        loc["lat"],
                "lng":        loc["lng"],
                "distance_m": round(dist, 1),
                "place_id":   pid,
            })

        # Pagination
        token = r.get("next_page_token")
        if not token or page >= 2:   # max 3 pages total -> 60 results
            break
        page += 1
        params = {"pagetoken": token, "key": API_KEY}
        time.sleep(2)   # Google enforces a short delay before the token activates

    return matches


## 7. Optional — Highlands store lookup

Only runs if `INCLUDE_STORE_LOOKUP = True` (off by default to save API cost).
Pulls rating / review count / hours / maps URL for each Highlands store.

In [10]:
FIND_PLACE_URL = "https://maps.googleapis.com/maps/api/place/findplacefromtext/json"
DETAILS_URL    = "https://maps.googleapis.com/maps/api/place/details/json"


def lookup_highlands(address, lat, lng):
    """Return a dict of Google place attributes for this Highlands store."""
    empty = {"Found on Google": "No", "Google Rating": "", "Review Count": "",
             "Price Level": "", "Opening Hours": "", "Google Maps URL": ""}
    if not INCLUDE_STORE_LOOKUP:
        return empty

    params = {
        "input":       f"Highlands Coffee {address}",
        "inputtype":   "textquery",
        "fields":      "place_id",
        "locationbias": f"point:{lat},{lng}",
        "key":         API_KEY,
    }
    r = requests.get(FIND_PLACE_URL, params=params).json()
    candidates = r.get("candidates", [])
    if not candidates:
        return empty

    pid = candidates[0].get("place_id")
    if not pid:
        return empty

    params = {
        "place_id": pid,
        "fields":   "name,rating,user_ratings_total,price_level,opening_hours",
        "key":      API_KEY,
    }
    d = requests.get(DETAILS_URL, params=params).json().get("result", {})
    hours = "; ".join(d.get("opening_hours", {}).get("weekday_text", []))
    return {
        "Found on Google":  "Yes",
        "Google Rating":    d.get("rating", ""),
        "Review Count":     d.get("user_ratings_total", ""),
        "Price Level":      d.get("price_level", ""),
        "Opening Hours":    hours,
        "Google Maps URL":  f"https://www.google.com/maps/place/?q=place_id:{pid}",
    }


## 8. Main loop — with checkpointing

Every 25 stores the current results are pickled to disk. If Colab
disconnects or you hit a rate limit, **re-run this cell** — it will skip
stores already processed and resume from where it stopped.

To start fresh, delete `checkpoint_500m.pkl` before running.

In [11]:
# ---- Load checkpoint if present --------------------------------------------
summary_rows = []
detail_rows  = []
done_stores  = set()

if Path(CHECKPOINT_FILE).exists():
    with open(CHECKPOINT_FILE, "rb") as f:
        state = pickle.load(f)
    summary_rows = state["summary_rows"]
    detail_rows  = state["detail_rows"]
    done_stores  = state["done_stores"]
    print(f"✅ Resuming from checkpoint — {len(done_stores)} stores already processed")
else:
    print("Starting fresh")


def save_checkpoint():
    with open(CHECKPOINT_FILE, "wb") as f:
        pickle.dump({
            "summary_rows": summary_rows,
            "detail_rows":  detail_rows,
            "done_stores":  done_stores,
        }, f)


# ---- Process each store ----------------------------------------------------
remaining = df[~df["Store"].isin(done_stores)].reset_index(drop=True)
print(f"{len(remaining)} stores to process\n")

for idx, row in tqdm(remaining.iterrows(), total=len(remaining)):
    store_name = row["Store"]
    lat, lng   = row["Latitude"], row["Longitude"]

    summary = {
        "Store":     store_name,
        "NS code":   row["NS code only"],
        "Address":   row["Address"],
        "Latitude":  lat,
        "Longitude": lng,
    }

    # Optional: look up the Highlands store itself
    summary.update(lookup_highlands(row["Address"], lat, lng))

    # Count each competitor
    total_competitors = 0
    for brand, variants in COMPETITORS.items():
        matches = nearby_brand_matches(lat, lng, RADIUS_M, brand, variants)
        summary[f"{brand} ({RADIUS_M}m)"] = len(matches)
        total_competitors += len(matches)

        for m in matches:
            detail_rows.append({
                "Highlands Store":  store_name,
                "NS code":          row["NS code only"],
                "Competitor Brand": brand,
                "Competitor Name":  m["name"],
                "Vicinity":         m["vicinity"],
                "Distance (m)":     m["distance_m"],
                "Competitor Lat":   m["lat"],
                "Competitor Lng":   m["lng"],
                "Place ID":         m["place_id"],
            })
        time.sleep(SLEEP_BETWEEN_CALLS)

    # Highlands self-check (subtract 1 for the store itself)
    self_matches = nearby_brand_matches(lat, lng, RADIUS_M,
                                        "Highlands Coffee", ["Highlands"])
    summary[f"Other Highlands ({RADIUS_M}m)"] = max(0, len(self_matches) - 1)
    summary[f"Total Competitors ({RADIUS_M}m)"] = total_competitors

    summary_rows.append(summary)
    done_stores.add(store_name)

    if (idx + 1) % CHECKPOINT_EVERY == 0:
        save_checkpoint()

save_checkpoint()
print(f"\n✅ Done — {len(summary_rows)} stores processed, "
      f"{len(detail_rows)} competitor matches recorded")


Starting fresh
996 stores to process



  0%|          | 0/996 [00:00<?, ?it/s]


✅ Done — 996 stores processed, 1750 competitor matches recorded


## 9. Write Excel output

In [12]:
summary_df = pd.DataFrame(summary_rows)
detail_df  = pd.DataFrame(detail_rows)

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
    detail_df.to_excel(writer, sheet_name="Details",  index=False)

print(f"✅ Wrote {OUTPUT_FILE}")
print(f"   Summary: {len(summary_df):,} rows × {len(summary_df.columns)} cols")
print(f"   Details: {len(detail_df):,} rows\n")

# Headline figures
competitor_cols = [f"{b} ({RADIUS_M}m)" for b in COMPETITORS]
print(f"Totals across all {len(summary_df)} Highlands stores (within {RADIUS_M}m):\n")
for c in competitor_cols:
    print(f"  {c:<35} {summary_df[c].sum():>6,}  "
          f"(avg {summary_df[c].mean():.2f} per store, "
          f"max {summary_df[c].max()})")


✅ Wrote Highlands_Competitor_500m.xlsx
   Summary: 996 rows × 20 cols
   Details: 1,750 rows

Totals across all 996 Highlands stores (within 500m):

  Starbucks (500m)                       343  (avg 0.34 per store, max 8)
  The Coffee House (500m)                271  (avg 0.27 per store, max 6)
  Phe La (500m)                          145  (avg 0.15 per store, max 3)
  Katinat (500m)                         252  (avg 0.25 per store, max 7)
  Cong (500m)                            101  (avg 0.10 per store, max 3)
  Trung Nguyen Legend (500m)             208  (avg 0.21 per store, max 4)
  Phuc Long (500m)                       430  (avg 0.43 per store, max 7)


## 10. Download

In [13]:
from google.colab import files
files.download(OUTPUT_FILE)
print("✅ Downloaded")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded


## 11. Quick spot-check (optional)

Look at the top 10 most-competed Highlands stores and the details behind
them. Useful to sanity-check a few numbers before trusting the full sheet.

In [14]:
top10 = summary_df.nlargest(10, f"Total Competitors ({RADIUS_M}m)")[
    ["Store", f"Total Competitors ({RADIUS_M}m)"] + competitor_cols
]
top10


,Store,Total Competitors (500m),Starbucks (500m),The Coffee House (500m),Phe La (500m),Katinat (500m),Cong (500m),Trung Nguyen Legend (500m),Phuc Long (500m)
774,HCSHCM0283 66 Nguyen Hue D1,23,8,0,1,7,1,1,5
76,HCSHCM0070 Saigon Centre 2,21,5,0,3,7,0,2,4
19,HCSHCM0034 FIDECO,20,5,0,1,7,0,0,7
833,HCSHCM0299 Ton That Thiep D1,20,6,0,1,7,0,1,5
14,HCSHCM0029 Vincom B3,18,6,0,1,4,2,3,2
144,HCSHCM0108 City Museum,18,4,0,2,5,2,3,2
267,HCSHCM0149 119 Ham Nghi,18,4,1,3,6,1,0,3
706,HCSHCM0212 Saigon Post Office,17,4,0,1,2,3,4,3
23,HCSHCM0004 Diamond,16,3,0,2,2,3,4,2
490,HCSHCM0201 Friendship Tower D1,15,4,1,1,2,3,2,2


In [15]:
# For the top store, show every competitor the script found
top_store = summary_df.nlargest(1, f"Total Competitors ({RADIUS_M}m)").iloc[0]["Store"]
print(f"Detail for: {top_store}\n")
detail_df[detail_df["Highlands Store"] == top_store].sort_values(
    ["Competitor Brand", "Distance (m)"]
)


Detail for: HCSHCM0283 66 Nguyen Hue D1



,Highlands Store,NS code,Competitor Brand,Competitor Name,Vicinity,Distance (m),Competitor Lat,Competitor Lng,Place ID
1422,HCSHCM0283 66 Nguyen Hue D1,HCSHCM0283,Cong,Cộng Cà Phê,"B3-10 Vincom, Đồng Khởi/72 Lê Thánh Tôn, Bến Nghé",449.0,10.778020,106.701990,ChIJh2KS2EcvdTERReyAHhqw6vo
1418,HCSHCM0283 66 Nguyen Hue D1,HCSHCM0283,Katinat,Katinat,"91 Đồng Khởi, Bến Nghé",66.7,10.774688,106.704278,ChIJj1y29UYvdTERjGrWH9smE-U
1415,HCSHCM0283 66 Nguyen Hue D1,HCSHCM0283,Katinat,KATINAT Oscar Nguyễn Huệ,"68A Nguyễn Huệ, Phường",66.9,10.774807,106.703332,ChIJqV9h-_MvdTER4H87N3c0m-o
1421,HCSHCM0283 66 Nguyen Hue D1,HCSHCM0283,Katinat,Katinat Huỳnh Thúc Kháng,"03 Huỳnh Thúc Kháng, Bến Nghé",85.2,10.773619,106.703626,ChIJSTRp8kIvdTERLVSpoVCRMwI
1417,HCSHCM0283 66 Nguyen Hue D1,HCSHCM0283,Katinat,KATINAT Nguyễn Huệ,"105 Nguyễn Huệ, Bến Nghé",94.5,10.773524,106.703763,ChIJvZ88zvkvdTERUkcEi0gzXmY
1419,HCSHCM0283 66 Nguyen Hue D1,HCSHCM0283,Katinat,KATINAT Takashimaya,"Toà nhà Saigon Centre, 92-94 Nam Kỳ Khởi Nghĩa...",371.1,10.772983,106.700670,ChIJEfn0pOMvdTERNFaTkMYBQoU
1416,HCSHCM0283 66 Nguyen Hue D1,HCSHCM0283,Katinat,Katinat – Bach Dang Wharf,10B Tôn Đức Thắng,374.3,10.775133,106.707096,ChIJkbXHKwAvdTER2nyVAWkOlww
1420,HCSHCM0283 66 Nguyen Hue D1,HCSHCM0283,Katinat,KATINAT Hàm Nghi,"120 Hàm Nghi, Bến Nghé",384.2,10.771146,106.702505,ChIJh_FRk_UvdTERoruyFlh1S2A
1414,HCSHCM0283 66 Nguyen Hue D1,HCSHCM0283,Phe La,Phê La - Hồ Tùng Mậu,"125 Hồ Tùng Mậu, Bến Nghé",136.0,10.773549,106.702839,ChIJCXB0uKUvdTERwgbFaAYnRI4
1427,HCSHCM0283 66 Nguyen Hue D1,HCSHCM0283,Phuc Long,Phuc Long Coffee & Tea Express,"63 Mạc Thị Bưởi, Bến Nghé",75.7,10.774697,106.704367,ChIJ1aH3YkYvdTERQS9GeWyH3ao
